# MNIST MLP3 — three-optimizer baseline comparison

This notebook compares the persisted three-seed results from SGD + momentum,
AdamW, and SGD + momentum + Muon. Run the three baseline notebooks first under
the same `RG_BASELINE_RUN_ROOT` (default: `baseline/runs/`).

It does **not retrain**. It rejects partial experiments by checking identical
seeds and shared settings, complete epoch/layer grids, every `final_state.pt`,
and every epoch checkpoint. It then writes trajectory summaries, final metrics,
convergence tables, paired seed-level contrasts, WeightWatcher/RG comparisons,
14 fixed-color plots, and a reproducibility manifest.

For cross-entropy $L$, classification perplexity is $\exp(L)$. On MNIST this is
an effective-class-count transform, not language-model perplexity. Paired
contrasts are always `optimizer_a - optimizer_b`; with only three seeds the
notebook reports Student-t intervals and does not claim high-powered asymptotic
significance tests.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "weightwatcher>=0.7.7"]
    )

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / "baseline"
    if (candidate / "rg_baselines").is_dir():
        ROOT = candidate
        break
    if (path / "rg_baselines").is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError("Run this notebook from a clone of CalculatedContent/rg_optimizers.")
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

raw = os.environ.get("RG_BASELINE_RUN_ROOT")
RUN_ROOT = (Path(raw).expanduser() if raw else ROOT / "runs")
if not RUN_ROOT.is_absolute():
    RUN_ROOT = Path.cwd() / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("baseline root:", ROOT)
print("shared run root:", RUN_ROOT)

## Validate and load persisted results

In [ ]:
from IPython.display import display
import pandas as pd
from rg_baselines.comparison import run_baseline_comparison

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 1000)

comparison = run_baseline_comparison(
    RUN_ROOT,
    output_dir=RUN_ROOT / "comparison",
    show_plots=True,
)
print(
    "Persistence audit passed:",
    3, "optimizers;",
    len(comparison.seeds), "seeds;",
    comparison.epochs, "epoch checkpoints per seed;",
    comparison.epochs + 1, "metric checkpoints including epoch 0.",
)
display(comparison.checkpoint_inventory)

## Final-epoch performance with 95% intervals

In [ ]:
FINAL_METRICS = [
    "test_accuracy", "test_loss", "test_perplexity",
    "train_accuracy", "train_loss",
    "accuracy_generalization_gap", "loss_generalization_gap",
    "parameter_l2_norm", "epoch_total_time_sec",
]
final_table = comparison.final_epoch_summary.loc[
    comparison.final_epoch_summary["metric"].isin(FINAL_METRICS),
    [
        "optimizer", "optimizer_label", "metric", "n", "mean",
        "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
    ],
].sort_values(["metric", "optimizer"])
display(final_table)

## Convergence and best-achieved performance

In [ ]:
CONVERGENCE_METRICS = [
    "best_test_accuracy", "best_test_accuracy_epoch",
    "final_test_accuracy", "final_test_loss", "final_test_perplexity",
    "epoch_to_test_accuracy_0.90", "epoch_to_test_accuracy_0.95",
    "epoch_to_test_accuracy_0.97", "epoch_to_test_accuracy_0.98",
]
display(
    comparison.convergence_summary.loc[
        comparison.convergence_summary["metric"].isin(CONVERGENCE_METRICS)
    ].sort_values(["metric", "optimizer"])
)

## Paired final-epoch optimizer contrasts

In [ ]:
display(
    comparison.paired_final_differences.sort_values(["metric", "contrast"])
)

## Persisted comparison package

In [ ]:
print("Comparison audit passed.")
print("comparison directory:", comparison.output_dir)
print("persisted comparison outputs:", len(comparison.expected_outputs))
display(pd.DataFrame({"path": [str(path) for path in comparison.expected_outputs]}))